# Criteo Uplift Analysis

This notebook analyzes a large randomized advertising experiment from Criteo (about 14 million users, roughly 85% treated). It covers a data audit, A/B test analysis (average treatment effect, confidence intervals, power, intent-to-treat vs. exposure effects), uplift modeling evaluated with Qini curves, and a targeting decision that estimates the value of showing ads only to the users most likely to respond.

**Inputs**

| Dataset | Files | Produced by |
|---|---|---|
| [arashnic/uplift-modeling](https://www.kaggle.com/datasets/arashnic/uplift-modeling) | `criteo-uplift-v2.1.csv` | Original data |
| `criteo-uplift-derived` | `criteo_uplift.parquet` | Section 0.3, with `RUN_DATA_PREP = True` |
| | `aipw_predictions.parquet` | Section 2.4, with `RUN_AIPW = True` |

Heavy steps run once; their outputs are saved to `criteo-uplift-derived`, and later runs load them with the flags set to `False`.

## 0. Setup

In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import duckdb
import sklearn
import lightgbm as lgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SEED = 42
np.random.seed(SEED)

print("pandas   ", pd.__version__)
print("numpy    ", np.__version__)
print("duckdb   ", duckdb.__version__)
print("sklearn  ", sklearn.__version__)
print("lightgbm ", lgb.__version__)

pandas    2.3.3
numpy     2.1.3
duckdb    1.3.2
sklearn   1.6.1
lightgbm  4.6.0


### 0.1 Locate input files

In [2]:
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024**2
        print(f"{path}  ({size_mb:,.1f} MB)")

/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv  (3,097.6 MB)
/kaggle/input/datasets/guilinzhao/criteo-uplift-derived/aipw_predictions.parquet  (195.9 MB)
/kaggle/input/datasets/guilinzhao/criteo-uplift-derived/criteo_uplift.parquet  (104.5 MB)


### 0.2 First look at the raw file

In [3]:
csv_files = sorted(glob.glob("/kaggle/input/**/*.csv*", recursive=True))
print(csv_files)

RAW_PATH = csv_files[0]
print("Using:", RAW_PATH)

sample = pd.read_csv(RAW_PATH, nrows=5)
print(sample.shape)
print(sample.dtypes)
sample

['/kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv']
Using: /kaggle/input/datasets/arashnic/uplift-modeling/criteo-uplift-v2.1.csv
(5, 16)
f0            float64
f1            float64
f2            float64
f3            float64
f4            float64
f5            float64
f6            float64
f7            float64
f8            float64
f9            float64
f10           float64
f11           float64
treatment       int64
conversion      int64
visit           int64
exposure        int64
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0


### 0.3 Convert CSV to Parquet (run once)

The raw CSV is about 3.1 GB. It is converted once to a compressed Parquet file with compact types (float32 features, int8 labels) and saved as a Kaggle dataset. Later runs set `RUN_DATA_PREP = False` and read the saved file directly.

In [4]:
import time

RUN_DATA_PREP = False  # Set to False after the Parquet file is saved as a Kaggle dataset

PARQUET_NAME = "criteo_uplift.parquet"
WORK_PATH = f"/kaggle/working/{PARQUET_NAME}"

FEATURE_COLS = [f"f{i}" for i in range(12)]
LABEL_COLS = ["treatment", "conversion", "visit", "exposure"]

if RUN_DATA_PREP:
    start = time.time()
    con = duckdb.connect()

    select_sql = ",\n        ".join(
        [f"CAST({c} AS FLOAT) AS {c}" for c in FEATURE_COLS]
        + [f"CAST({c} AS TINYINT) AS {c}" for c in LABEL_COLS]
    )

    con.execute(f"""
        COPY (
            SELECT
        {select_sql}
            FROM read_csv('{RAW_PATH}', header = true)
        ) TO '{WORK_PATH}' (FORMAT parquet, COMPRESSION zstd)
    """)

    raw_rows = con.execute(
        f"SELECT COUNT(*) FROM read_csv('{RAW_PATH}', header = true)"
    ).fetchall()[0][0]
    pq_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{WORK_PATH}')"
    ).fetchall()[0][0]
    con.close()

    print(f"Raw CSV rows:     {raw_rows:,}")
    print(f"Parquet rows:     {pq_rows:,}")
    print(f"Row counts match: {raw_rows == pq_rows}")
    print(f"Parquet size:     {os.path.getsize(WORK_PATH) / 1024**2:,.1f} MB")
    print(f"Elapsed:          {time.time() - start:,.1f} s")
else:
    print("Skipping data prep; the saved Parquet will be loaded below.")

Skipping data prep; the saved Parquet will be loaded below.


### 0.4 Load the prepared data

In [5]:
if RUN_DATA_PREP:
    DATA_PATH = WORK_PATH
else:
    found = glob.glob(f"/kaggle/input/**/{PARQUET_NAME}", recursive=True)
    assert found, "Prepared Parquet not found. Attach the prepared dataset as an input."
    DATA_PATH = found[0]

df = pd.read_parquet(DATA_PATH)

print("Loaded from:", DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
print(df.dtypes)
df.head()

Loaded from: /kaggle/input/datasets/guilinzhao/criteo-uplift-derived/criteo_uplift.parquet
Shape: 13,979,592 rows x 16 columns
Memory: 693.3 MB
f0            float32
f1            float32
f2            float32
f3            float32
f4            float32
f5            float32
f6            float32
f7            float32
f8            float32
f9            float32
f10           float32
f11           float32
treatment        int8
conversion       int8
visit            int8
exposure         int8
dtype: object


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.6164,10.0597,8.9764,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
1,12.6164,10.0597,9.0027,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
2,12.6164,10.0597,8.9648,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
3,12.6164,10.0597,9.0028,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0
4,12.6164,10.0597,9.0380,4.6799,10.2805,4.1155,0.2944,4.8338,3.9554,13.1901,5.3004,-0.1687,1,0,0,0


## 1. Data audit

Before estimating any effect, the data is checked for completeness, valid label values, consistency between labels, and whether treatment assignment looks random.

### 1.1 Missing values, label values, and duplicates

In [6]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nDistinct values in label columns:")
for c in LABEL_COLS:
    print(f"  {c:<11}", sorted(df[c].unique().tolist()))

n_dup = df.duplicated().sum()
n_dup_feat = df.duplicated(subset=FEATURE_COLS).sum()
print(f"\nExact duplicate rows (all 16 columns): {n_dup:,} ({n_dup / len(df):.2%})")
print(f"Rows with duplicated features only:    {n_dup_feat:,} ({n_dup_feat / len(df):.2%})")

Missing values per column:
None

Distinct values in label columns:
  treatment   [0, 1]
  conversion  [0, 1]
  visit       [0, 1]
  exposure    [0, 1]

Exact duplicate rows (all 16 columns): 1,655,747 (11.84%)
Rows with duplicated features only:    2,120,151 (15.17%)


### 1.2 Group sizes and outcome rates

In [7]:
def group_summary(data):
    return pd.Series({
        "users": len(data),
        "visit_rate": data["visit"].mean(),
        "conversion_rate": data["conversion"].mean(),
        "exposure_rate": data["exposure"].mean(),
    })

summary = df.groupby("treatment").apply(group_summary)
summary.index = summary.index.map({0: "control", 1: "treatment"})
summary.loc["overall"] = group_summary(df)
summary["share"] = summary["users"] / len(df)

summary.style.format({
    "users": "{:,.0f}",
    "share": "{:.2%}",
    "visit_rate": "{:.3%}",
    "conversion_rate": "{:.4%}",
    "exposure_rate": "{:.3%}",
})

,users,visit_rate,conversion_rate,exposure_rate,share
treatment,,,,,
control,"2,096,937",3.820%,0.1938%,0.000%,15.00%
treatment,"11,882,655",4.854%,0.3089%,3.604%,85.00%
overall,"13,979,592",4.699%,0.2917%,3.063%,100.00%


### 1.3 Logical consistency between labels

Expected relationships: control users should never be exposed to the ad, and a conversion should normally follow a visit.

In [8]:
print("Treatment x exposure:")
print(pd.crosstab(df["treatment"], df["exposure"], margins=True), "\n")

print("Visit x conversion:")
print(pd.crosstab(df["visit"], df["conversion"], margins=True), "\n")

checks = {
    "Control users with exposure = 1": ((df["treatment"] == 0) & (df["exposure"] == 1)).sum(),
    "Conversions without a visit": ((df["conversion"] == 1) & (df["visit"] == 0)).sum(),
}
for name, n in checks.items():
    print(f"{name:<34} {n:,}")

treated = df[df["treatment"] == 1]
print(f"\nShare of treated users actually exposed: {treated['exposure'].mean():.2%}")

Treatment x exposure:
exposure          0       1       All
treatment                            
0           2096937       0   2096937
1          11454443  428212  11882655
All        13551380  428212  13979592 

Visit x conversion:
conversion         0      1       All
visit                                
0           13322663      0  13322663
1             616155  40774    656929
All         13938818  40774  13979592 

Control users with exposure = 1    0
Conversions without a visit        0

Share of treated users actually exposed: 3.60%


### 1.4 Randomization check: covariate balance

If assignment is random, feature distributions should be nearly identical across groups. With 14 million rows, even tiny differences are statistically significant, so balance is judged by the standardized mean difference (SMD) rather than p-values. An absolute SMD below 0.1 is a common threshold for good balance.

In [9]:
is_treated = df["treatment"] == 1
rows = []
for c in FEATURE_COLS:
    x = df[c].astype("float64")
    m1, m0 = x[is_treated].mean(), x[~is_treated].mean()
    v1, v0 = x[is_treated].var(), x[~is_treated].var()
    rows.append({
        "feature": c,
        "mean_treatment": m1,
        "mean_control": m0,
        "smd": (m1 - m0) / np.sqrt((v1 + v0) / 2),
    })

balance = pd.DataFrame(rows).set_index("feature")
balance["abs_smd"] = balance["smd"].abs()
print(f"Max |SMD|: {balance['abs_smd'].max():.4f} ({balance['abs_smd'].idxmax()})")
print(f"Features with |SMD| >= 0.1: {(balance['abs_smd'] >= 0.1).sum()}")
balance.sort_values("abs_smd", ascending=False)

Max |SMD|: 0.0488 (f3)
Features with |SMD| >= 0.1: 0


,mean_treatment,mean_control,smd,abs_smd
feature,,,,
f3,4.1694,4.2328,-0.0488,0.0488
f6,-4.1828,-3.9999,-0.0404,0.0404
f5,4.0266,4.0393,-0.0306,0.0306
f9,16.0526,15.8863,0.0240,0.0240
f1,10.0703,10.0679,0.0240,0.0240
f8,3.9334,3.9347,-0.0224,0.0224
f7,5.1056,5.0803,0.0213,0.0213
f10,5.3337,5.3319,0.0106,0.0106
f4,10.3392,10.3365,0.0080,0.0080


### 1.5 Randomization check: can features predict treatment?

A stronger test: train a classifier to predict treatment from the 12 features. Under true randomization, it should do no better than chance (ROC-AUC close to 0.5). A 1 million-row random sample is enough for this check.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sample = df.sample(n=1_000_000, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    sample[FEATURE_COLS], sample["treatment"],
    test_size=0.3, stratify=sample["treatment"], random_state=SEED,
)

clf = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.1, num_leaves=31,
    random_state=SEED, verbose=-1,
)
clf.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
print(f"ROC-AUC for predicting treatment from features: {auc:.4f}")

ROC-AUC for predicting treatment from features: 0.5126


### 1.6 Permutation baseline for the treatment classifier

To judge whether an AUC of about 0.51 is meaningful, the same classifier is trained on randomly shuffled treatment labels. The spread of these AUCs shows what pure chance looks like at this sample size.

In [11]:
rng = np.random.default_rng(SEED)
perm_aucs = []
for i in range(3):
    y_tr_perm = rng.permutation(y_tr.values)
    y_te_perm = rng.permutation(y_te.values)
    clf_perm = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, num_leaves=31,
        random_state=SEED, verbose=-1,
    )
    clf_perm.fit(X_tr, y_tr_perm)
    perm_aucs.append(roc_auc_score(y_te_perm, clf_perm.predict_proba(X_te)[:, 1]))

print(f"Real-label AUC:      {auc:.4f}")
print("Shuffled-label AUCs:", [round(float(a), 4) for a in perm_aucs])

importance = pd.Series(
    clf.booster_.feature_importance(importance_type="gain"), index=FEATURE_COLS
)
print("\nFeatures that best separate treatment from control (gain share):")
print((importance / importance.sum()).sort_values(ascending=False).round(3))

Real-label AUC:      0.5126
Shuffled-label AUCs: [0.4977, 0.5009, 0.4993]

Features that best separate treatment from control (gain share):
f0    0.1670
f2    0.1610
f6    0.1490
f8    0.1480
f9    0.1040
f10   0.0740
f7    0.0730
f3    0.0630
f4    0.0290
f5    0.0150
f11   0.0110
f1    0.0080
dtype: float64


### 1.7 Audit summary

- **Complete and valid.** There are no missing values, and all four labels are binary. About 11.8% of rows are exact duplicates and 15.2% share identical feature values. Because the features are anonymized projections, different users can share identical values, so all rows are kept.
- **Group split.** 85.00% of users are in treatment and 15.00% in control, within 2 users of an exact 85/15 split. The dataset was sub-sampled by its publisher, so a sample ratio mismatch test is not informative here.
- **Labels are consistent.** No control user was exposed to the ad, and every conversion has a visit. Only 3.60% of treated users were actually exposed, so a treatment-vs-control comparison measures the effect of being assigned to treatment (intent-to-treat), not the effect of seeing the ad.
- **Randomization holds, with a small imbalance.** All absolute SMDs are below 0.1 (maximum 0.049 for `f3`), and a classifier predicts treatment with ROC-AUC 0.513. With shuffled labels the AUC stays between 0.498 and 0.501, so the imbalance is small but larger than chance. It is spread across several features rather than concentrated in one. The publisher states that the data was assembled from several incrementality tests, and pooling tests with different populations is a plausible source. Section 2 checks whether this matters by comparing the raw estimate with a regression-adjusted one.
- **Absolute levels are not Criteo's real results.** The publisher sub-sampled the data non-uniformly for privacy, so the original incrementality level cannot be recovered. Effect sizes and value estimates in this notebook describe this dataset, not Criteo's actual campaigns; comparisons between methods within the dataset remain meaningful.

## 2. A/B test analysis

This section estimates the effect of the ad campaign on visits and conversions. It starts with the intent-to-treat effect of being assigned to treatment, checks robustness with regression adjustment, evaluates statistical power, and finally estimates the effect on users who actually saw the ad. Differences between rates are reported in percentage points.

### 2.1 Average treatment effect (intent-to-treat)

The intent-to-treat (ITT) effect compares everyone assigned to treatment with everyone assigned to control, regardless of whether they saw an ad. Because assignment is random, this difference is an unbiased estimate of the effect of assignment. Confidence intervals use the normal approximation, which is accurate at this sample size; the interval for relative lift uses the delta method on the log ratio.

In [12]:
from scipy import stats

Z = stats.norm.ppf(0.975)
OUTCOMES = ["visit", "conversion"]

def diff_in_means(data, outcome, group="treatment"):
    y1 = data.loc[data[group] == 1, outcome]
    y0 = data.loc[data[group] == 0, outcome]
    n1, n0 = len(y1), len(y0)
    p1, p0 = y1.mean(), y0.mean()

    diff = p1 - p0
    se = np.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)

    log_ratio = np.log(p1 / p0)
    se_log = np.sqrt((1 - p1) / (n1 * p1) + (1 - p0) / (n0 * p0))

    return {
        "outcome": outcome,
        "rate_control": p0,
        "rate_treatment": p1,
        "ate": diff,
        "se": se,
        "ci_low": diff - Z * se,
        "ci_high": diff + Z * se,
        "z": diff / se,
        "p_value": 2 * stats.norm.sf(abs(diff / se)),
        "rel_lift": p1 / p0 - 1,
        "rel_ci_low": np.exp(log_ratio - Z * se_log) - 1,
        "rel_ci_high": np.exp(log_ratio + Z * se_log) - 1,
    }

itt = pd.DataFrame([diff_in_means(df, y) for y in OUTCOMES]).set_index("outcome")

itt.style.format({
    "rate_control": "{:.4%}", "rate_treatment": "{:.4%}",
    "ate": "{:.4%}", "se": "{:.4%}", "ci_low": "{:.4%}", "ci_high": "{:.4%}",
    "z": "{:.1f}", "p_value": "{:.2e}",
    "rel_lift": "{:.1%}", "rel_ci_low": "{:.1%}", "rel_ci_high": "{:.1%}",
})

,rate_control,rate_treatment,ate,se,ci_low,ci_high,z,p_value,rel_lift,rel_ci_low,rel_ci_high
outcome,,,,,,,,,,,
visit,3.8201%,4.8543%,1.0342%,0.0146%,1.0056%,1.0629%,70.7,0.00e+00,27.1%,26.2%,28.0%
conversion,0.1938%,0.3089%,0.1152%,0.0034%,0.1085%,0.1219%,33.5,3.19e-246,59.4%,54.4%,64.7%


### 2.2 Regression-adjusted estimate

The audit found a small but real feature imbalance between groups. To check whether it affects the estimate, each outcome is regressed on the treatment indicator and the 12 centered features (a linear probability model) with heteroskedasticity-robust standard errors. If the adjusted estimate is close to the raw difference, the imbalance does not matter for the average effect. Adjustment also tends to shrink the standard error, because the features explain part of the outcome variance.

In [13]:
def regression_adjusted_ate(data, outcome, features=FEATURE_COLS, group="treatment"):
    n = len(data)
    Xf = data[features].to_numpy(dtype=np.float64)
    Xf -= Xf.mean(axis=0)
    X = np.column_stack([np.ones(n), data[group].to_numpy(dtype=np.float64), Xf])
    del Xf
    y = data[outcome].to_numpy(dtype=np.float64)

    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta
    meat = (X * (resid ** 2)[:, None]).T @ X
    cov = XtX_inv @ meat @ XtX_inv  # HC0 robust covariance
    return beta[1], np.sqrt(cov[1, 1])

start = time.time()
rows = []
for y in OUTCOMES:
    adj, adj_se = regression_adjusted_ate(df, y)
    raw, raw_se = itt.loc[y, "ate"], itt.loc[y, "se"]
    rows.append({
        "outcome": y,
        "ate_raw": raw,
        "se_raw": raw_se,
        "ate_adjusted": adj,
        "se_adjusted": adj_se,
        "ci_low_adj": adj - Z * adj_se,
        "ci_high_adj": adj + Z * adj_se,
        "change_in_ate": adj / raw - 1,
        "se_reduction": 1 - adj_se / raw_se,
    })
print(f"Elapsed: {time.time() - start:,.1f} s")

adjusted = pd.DataFrame(rows).set_index("outcome")
adjusted.style.format({
    "ate_raw": "{:.4%}", "se_raw": "{:.4%}",
    "ate_adjusted": "{:.4%}", "se_adjusted": "{:.4%}",
    "ci_low_adj": "{:.4%}", "ci_high_adj": "{:.4%}",
    "change_in_ate": "{:+.1%}", "se_reduction": "{:.1%}",
})

Elapsed: 7.2 s


,ate_raw,se_raw,ate_adjusted,se_adjusted,ci_low_adj,ci_high_adj,change_in_ate,se_reduction
outcome,,,,,,,,
visit,1.0342%,0.0146%,0.6982%,0.0126%,0.6735%,0.7230%,-32.5%,13.7%
conversion,0.1152%,0.0034%,0.0916%,0.0032%,0.0853%,0.0979%,-20.5%,6.1%


### 2.3 Why does adjustment move the estimate?

In a clean randomized experiment, adjustment should change the standard error but barely move the estimate. Here it lowers the visit effect by about a third and the conversion effect by about a fifth, and the raw and adjusted confidence intervals do not overlap. For an OLS model with an intercept, a treatment indicator, and features, the gap is exact: raw difference minus adjusted effect equals the sum over features of (treatment mean minus control mean) times the feature's coefficient. This decomposes the gap feature by feature.

In [14]:
def ols_coefficients(data, outcome, features=FEATURE_COLS, group="treatment"):
    n = len(data)
    Xf = data[features].to_numpy(dtype=np.float64)
    Xf -= Xf.mean(axis=0)
    X = np.column_stack([np.ones(n), data[group].to_numpy(dtype=np.float64), Xf])
    del Xf
    y = data[outcome].to_numpy(dtype=np.float64)
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    return pd.Series(beta[2:], index=features)

mean_gap = balance["mean_treatment"] - balance["mean_control"]

for y in OUTCOMES:
    coefs = ols_coefficients(df, y)
    contrib = (mean_gap * coefs).rename("contribution")
    gap = itt.loc[y, "ate"] - adjusted.loc[y, "ate_adjusted"]
    print(f"--- {y} ---")
    print(f"Raw minus adjusted:         {gap:.5%}")
    print(f"Sum of feature contributions: {contrib.sum():.5%}")
    out = pd.DataFrame({
        "mean_gap": mean_gap,
        "coefficient": coefs,
        "contribution": contrib,
        "share_of_gap": contrib / gap,
    }).sort_values("contribution", key=abs, ascending=False)
    display(out.style.format({
        "mean_gap": "{:+.4f}", "coefficient": "{:+.5f}",
        "contribution": "{:+.5%}", "share_of_gap": "{:+.1%}",
    }))

--- visit ---
Raw minus adjusted:         0.33602%
Sum of feature contributions: 0.33602%


,mean_gap,coefficient,contribution,share_of_gap
f8,-0.0013,-1.09740,+0.13826%,+41.1%
f9,+0.1663,+0.00703,+0.11688%,+34.8%
f6,-0.1829,-0.00165,+0.03026%,+9.0%
f2,-0.0019,-0.09102,+0.01704%,+5.1%
f3,-0.0634,-0.00222,+0.01406%,+4.2%
f5,-0.0127,-0.00795,+0.01013%,+3.0%
f4,+0.0027,+0.02815,+0.00766%,+2.3%
f0,-0.0369,-0.00053,+0.00196%,+0.6%
f1,+0.0024,-0.00728,-0.00175%,-0.5%
f11,-0.0001,-0.14472,+0.00169%,+0.5%


--- conversion ---
Raw minus adjusted:         0.02357%
Sum of feature contributions: 0.02357%


,mean_gap,coefficient,contribution,share_of_gap
f4,+0.0027,+0.03959,+0.01077%,+45.7%
f10,+0.0018,-0.03510,-0.00618%,-26.2%
f3,-0.0634,-0.00083,+0.00529%,+22.4%
f11,-0.0001,-0.33788,+0.00396%,+16.8%
f9,+0.1663,+0.00018,+0.00300%,+12.7%
f5,-0.0127,-0.00216,+0.00275%,+11.6%
f6,-0.1829,-0.00013,+0.00241%,+10.2%
f1,+0.0024,+0.00415,+0.00100%,+4.2%
f8,-0.0013,-0.00774,+0.00098%,+4.1%
f7,+0.0253,-0.00028,-0.00072%,-3.0%


### 2.4 Doubly robust estimate (AIPW with cross-fitting)

The linear adjustment may not fully capture how features relate to outcomes. As a more flexible check, the average effect is estimated with augmented inverse probability weighting (AIPW). It combines a propensity model (probability of treatment given features) with outcome models for each group, and stays consistent if either one is correct. All models are LightGBM, fitted with 2-fold cross-fitting so that each user's predictions come from models trained on the other half of the data. The data is shuffled before splitting because the raw file is not in random order.

In [15]:
from sklearn.model_selection import KFold

RUN_AIPW = False  # Set to True to refit the cross-fitted models (about 25 minutes)
AIPW_NAME = "aipw_predictions.parquet"

def lgb_clf():
    return lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        min_child_samples=500, random_state=SEED, verbose=-1,
    )

T_all = df["treatment"].to_numpy()
n = len(df)

if RUN_AIPW:
    X_all = df[FEATURE_COLS]
    e_hat = np.zeros(n)
    m1_hat = {y: np.zeros(n) for y in OUTCOMES}
    m0_hat = {y: np.zeros(n) for y in OUTCOMES}

    start = time.time()
    kf = KFold(n_splits=2, shuffle=True, random_state=SEED)
    for fold, (tr, te) in enumerate(kf.split(X_all), start=1):
        X_tr, X_te = X_all.iloc[tr], X_all.iloc[te]
        T_tr = T_all[tr]

        prop = lgb_clf().fit(X_tr, T_tr)
        e_hat[te] = prop.predict_proba(X_te)[:, 1]

        for y in OUTCOMES:
            Y_tr = df[y].to_numpy()[tr]
            m1 = lgb_clf().fit(X_tr[T_tr == 1], Y_tr[T_tr == 1])
            m0 = lgb_clf().fit(X_tr[T_tr == 0], Y_tr[T_tr == 0])
            m1_hat[y][te] = m1.predict_proba(X_te)[:, 1]
            m0_hat[y][te] = m0.predict_proba(X_te)[:, 1]
        print(f"Fold {fold} done, {time.time() - start:,.0f} s elapsed")

    pd.DataFrame({
        "e_hat": e_hat,
        "m1_visit": m1_hat["visit"], "m0_visit": m0_hat["visit"],
        "m1_conversion": m1_hat["conversion"], "m0_conversion": m0_hat["conversion"],
    }).astype("float32").to_parquet(f"/kaggle/working/{AIPW_NAME}", compression="zstd")
else:
    found = glob.glob(f"/kaggle/input/**/{AIPW_NAME}", recursive=True)
    assert found, "AIPW predictions not found. Attach the updated dataset or set RUN_AIPW = True."
    preds = pd.read_parquet(found[0])
    e_hat = preds["e_hat"].to_numpy(dtype=np.float64)
    m1_hat = {y: preds[f"m1_{y}"].to_numpy(dtype=np.float64) for y in OUTCOMES}
    m0_hat = {y: preds[f"m0_{y}"].to_numpy(dtype=np.float64) for y in OUTCOMES}
    del preds
    print("Loaded cross-fitted predictions from:", found[0])

e_clip = np.clip(e_hat, 0.01, 0.99)
print(f"Propensity range: {e_hat.min():.3f} to {e_hat.max():.3f}")

Loaded cross-fitted predictions from: /kaggle/input/datasets/guilinzhao/criteo-uplift-derived/aipw_predictions.parquet
Propensity range: 0.644 to 0.993


In [16]:
def aipw_scores(y):
    Y = df[y].to_numpy().astype(np.float64)
    return (
        m1_hat[y] - m0_hat[y]
        + T_all * (Y - m1_hat[y]) / e_clip
        - (1 - T_all) * (Y - m0_hat[y]) / (1 - e_clip)
    )

rows = []
for y in OUTCOMES:
    psi = aipw_scores(y)
    ate_dr = psi.mean()
    se_dr = psi.std(ddof=1) / np.sqrt(n)
    rows.append({
        "outcome": y,
        "ate_raw": itt.loc[y, "ate"],
        "ate_linear_adj": adjusted.loc[y, "ate_adjusted"],
        "ate_outcome_model": (m1_hat[y] - m0_hat[y]).mean(),
        "ate_aipw": ate_dr,
        "se_aipw": se_dr,
        "ci_low_aipw": ate_dr - Z * se_dr,
        "ci_high_aipw": ate_dr + Z * se_dr,
        "rel_lift_aipw": ate_dr / (itt.loc[y, "rate_treatment"] - ate_dr),
    })

dr = pd.DataFrame(rows).set_index("outcome")
dr.style.format("{:.4%}")

,ate_raw,ate_linear_adj,ate_outcome_model,ate_aipw,se_aipw,ci_low_aipw,ci_high_aipw,rel_lift_aipw
outcome,,,,,,,,
visit,1.0342%,0.6982%,0.7453%,0.7483%,0.0129%,0.7230%,0.7736%,18.2244%
conversion,0.1152%,0.0916%,0.1115%,0.1004%,0.0036%,0.0934%,0.1074%,48.1548%


### 2.5 Power and minimum detectable effect

Power analysis asks how small an effect this experiment could reliably detect. The minimum detectable effect (MDE) is computed at a 5% two-sided significance level and 80% power. The second table shows how the size of the control group changes the MDE for conversion when the total sample is held fixed, which is the practical trade-off when an advertiser decides how many users to hold out.

In [17]:
Z_ALPHA = stats.norm.ppf(0.975)
Z_POWER = stats.norm.ppf(0.80)

n_total = len(df)
n1 = int((df["treatment"] == 1).sum())
n0 = n_total - n1

def mde(p0, n1, n0):
    return (Z_ALPHA + Z_POWER) * np.sqrt(p0 * (1 - p0) * (1 / n1 + 1 / n0))

def required_total_n(p0, effect, control_share):
    k = 1 / (1 - control_share) + 1 / control_share
    return (Z_ALPHA + Z_POWER) ** 2 * p0 * (1 - p0) * k / effect ** 2

rows = []
for y in OUTCOMES:
    p0 = itt.loc[y, "rate_control"]
    effect = itt.loc[y, "ate"]
    m = mde(p0, n1, n0)
    rows.append({
        "outcome": y,
        "baseline_rate": p0,
        "mde_abs": m,
        "mde_rel": m / p0,
        "observed_ate": effect,
        "observed_over_mde": effect / m,
        "n_needed_85_15": required_total_n(p0, effect, n0 / n_total),
    })
power = pd.DataFrame(rows).set_index("outcome")
display(power.style.format({
    "baseline_rate": "{:.4%}", "mde_abs": "{:.4%}", "mde_rel": "{:.1%}",
    "observed_ate": "{:.4%}", "observed_over_mde": "{:.1f}x",
    "n_needed_85_15": "{:,.0f}",
}))

p0_conv = itt.loc["conversion", "rate_control"]
ate_conv = itt.loc["conversion", "ate"]
rows = []
for share in [0.01, 0.05, 0.10, 0.15, 0.30, 0.50]:
    nc = n_total * share
    nt = n_total - nc
    m = mde(p0_conv, nt, nc)
    se = np.sqrt(p0_conv * (1 - p0_conv) * (1 / nt + 1 / nc))
    rows.append({
        "control_share": share,
        "mde_abs": m,
        "mde_rel": m / p0_conv,
        "power_for_observed_effect": stats.norm.cdf(abs(ate_conv) / se - Z_ALPHA),
    })
holdout = pd.DataFrame(rows).set_index("control_share")
holdout.style.format({
    "mde_abs": "{:.4%}", "mde_rel": "{:.1%}", "power_for_observed_effect": "{:.1%}",
})

,baseline_rate,mde_abs,mde_rel,observed_ate,observed_over_mde,n_needed_85_15
outcome,,,,,,
visit,3.8201%,0.0402%,1.1%,1.0342%,25.7x,"21,145"
conversion,0.1938%,0.0092%,4.8%,0.1152%,12.5x,"89,724"


,mde_abs,mde_rel,power_for_observed_effect
control_share,,,
0.010000,0.0331%,17.1%,100.0%
0.050000,0.0151%,7.8%,100.0%
0.100000,0.0110%,5.7%,100.0%
0.150000,0.0092%,4.8%,100.0%
0.300000,0.0072%,3.7%,100.0%
0.500000,0.0066%,3.4%,100.0%


### 2.6 Effect of actually seeing the ad

Only a small share of treated users were exposed, so the ITT effect is diluted across many users who never saw an ad. Two estimates are compared:

- **Naive comparison:** exposed treated users vs. the whole control group. This is biased, because users who end up exposed are likely more active and would convert more often even without the ad.
- **Effect on exposed users (complier average causal effect):** the ITT effect divided by the exposure rate in treatment (the Wald estimator). It relies on two assumptions: control users could not be exposed (verified in the audit), and assignment affects outcomes only through exposure. Under these assumptions, exposed treated users are exactly the "compliers", and their counterfactual rate without the ad can be recovered from the control group.

In [18]:
t = df[df["treatment"] == 1]
c = df[df["treatment"] == 0]
pi = t["exposure"].mean()
n_t, n_c = len(t), len(c)
print(f"Exposure rate in treatment (compliance rate): {pi:.4%}\n")

rows = []
for y in OUTCOMES:
    itt_y = itt.loc[y, "ate"]
    var_itt = itt.loc[y, "se"] ** 2
    var_pi = pi * (1 - pi) / n_t
    cov_itt_pi = np.cov(t[y].astype("float64"), t["exposure"].astype("float64"))[0, 1] / n_t

    cace = itt_y / pi
    se_cace = np.sqrt(
        var_itt / pi**2 + itt_y**2 * var_pi / pi**4 - 2 * itt_y * cov_itt_pi / pi**3
    )

    rate_exposed = t.loc[t["exposure"] == 1, y].mean()
    rate_unexposed_treated = t.loc[t["exposure"] == 0, y].mean()
    rate_control = c[y].mean()
    rate_compliers_without_ad = (rate_control - (1 - pi) * rate_unexposed_treated) / pi

    rows.append({
        "outcome": y,
        "rate_control": rate_control,
        "rate_exposed": rate_exposed,
        "rate_unexposed_treated": rate_unexposed_treated,
        "naive_effect": rate_exposed - rate_control,
        "itt": itt_y,
        "cace": cace,
        "cace_ci_low": cace - Z * se_cace,
        "cace_ci_high": cace + Z * se_cace,
        "exposed_rate_without_ad": rate_compliers_without_ad,
        "cace_rel_lift": rate_exposed / rate_compliers_without_ad - 1,
    })

exposure_effects = pd.DataFrame(rows).set_index("outcome")
exposure_effects.T.style.format("{:.4%}")

Exposure rate in treatment (compliance rate): 3.6037%



outcome,visit,conversion
rate_control,3.8201%,0.1938%
rate_exposed,41.4538%,5.3784%
rate_unexposed_treated,3.4861%,0.1194%
naive_effect,37.6337%,5.1847%
itt,1.0342%,0.1152%
cace,28.6996%,3.1964%
cace_ci_low,27.9112%,3.0100%
cace_ci_high,29.4880%,3.3828%
exposed_rate_without_ad,12.7541%,2.1820%
cace_rel_lift,225.0218%,146.4872%


The estimates above use the raw ITT effect, which the adjustment analysis showed is biased upward. The adjusted version below divides the AIPW effect by an inverse-probability-weighted estimate of the share of users who would be exposed if assigned to treatment. Both numerator and denominator therefore account for the feature imbalance, and the confidence interval comes from the influence function of the ratio.

In [19]:
D_all = df["exposure"].to_numpy().astype(np.float64)
w_t = T_all / e_clip
psi_d = w_t * D_all
mu_d = psi_d.mean()
print(f"Exposure share if treated, raw:      {pi:.4%}")
print(f"Exposure share if treated, adjusted: {mu_d:.4%}\n")

rows = []
for y in OUTCOMES:
    Y = df[y].to_numpy().astype(np.float64)
    psi_y = aipw_scores(y)
    cace = psi_y.mean() / mu_d
    psi_ratio = (psi_y - cace * psi_d) / mu_d
    se = psi_ratio.std(ddof=1) / np.sqrt(n)

    rate_with_ad = np.mean(w_t * D_all * Y) / mu_d
    rate_without_ad = rate_with_ad - cace

    rows.append({
        "outcome": y,
        "naive_effect": exposure_effects.loc[y, "naive_effect"],
        "cace_raw": exposure_effects.loc[y, "cace"],
        "cace_adjusted": cace,
        "ci_low": cace - Z * se,
        "ci_high": cace + Z * se,
        "exposed_rate_with_ad": rate_with_ad,
        "exposed_rate_without_ad": rate_without_ad,
        "rel_lift_exposed": cace / rate_without_ad,
    })

cace_adj = pd.DataFrame(rows).set_index("outcome")
cace_adj.T.style.format("{:.4%}")

Exposure share if treated, raw:      3.6037%
Exposure share if treated, adjusted: 3.5264%



outcome,visit,conversion
naive_effect,37.6337%,5.1847%
cace_raw,28.6996%,3.1964%
cace_adjusted,21.2201%,2.8476%
ci_low,20.5071%,2.6487%
ci_high,21.9331%,3.0465%
exposed_rate_with_ad,41.1820%,5.3920%
exposed_rate_without_ad,19.9619%,2.5444%
rel_lift_exposed,106.3032%,111.9178%


### 2.7 Summary

- **The raw comparison overstates the effect.** Assignment to treatment raised the visit rate by 1.03 percentage points and the conversion rate by 0.115 points in a simple difference of means. Adjusting for features moved both estimates down substantially, even though no feature had an absolute SMD above 0.05. The decomposition shows why: tiny mean differences in features that strongly predict outcomes (for visits, mainly `f8` and `f9`) add up to a meaningful bias.
- **The doubly robust estimate is the main result.** AIPW with cross-fitted LightGBM models estimates that assignment to treatment raised visits by **0.75 points** (95% CI 0.72 to 0.77, +18% relative) and conversions by **0.10 points** (95% CI 0.093 to 0.107, +48% relative). The raw difference overstates these effects by about 38% and 15%. Propensity scores range from 0.64 to 0.99, so overlap between groups is good. The linear adjustment lands slightly below AIPW, consistent with it missing non-linear relationships between features and outcomes.
- **The experiment had far more power than needed.** The minimum detectable effect for conversion is 0.009 points (4.8% relative), about one eleventh of the estimated effect. Moving from an 85/15 to a 50/50 split would only lower the MDE to 3.4% relative, and even a 1% holdout would detect an effect of this size, so the small control group cost very little here.
- **Users who saw the ad roughly doubled their conversion rate.** Only about 3.5% of treated users were exposed. Dividing the AIPW effect by the adjusted exposure share gives an effect on exposed users of **2.85 points** for conversion (95% CI 2.65 to 3.05) and 21.2 points for visits. Without the ad, these users would have converted at about 2.5%, so the ad lifted their conversion rate by roughly 110%.
- **Comparing exposed users with the control group is misleading.** That naive comparison gives 5.18 points for conversion, about 80% too high. Exposed users would have converted about 13 times more often than the average control user even without the ad, because ads tend to be shown to users who are already likely to buy. The exposure estimate assumes that assignment affects outcomes only through exposure and that control users could not be exposed, which the audit confirmed.
- **Absolute levels are dataset-specific.** As noted in the audit, the publisher sub-sampled the data non-uniformly, so these magnitudes describe this dataset rather than Criteo's actual campaigns.